# Preprocesamiento de Datos para Machine Learning

El **preprocesamiento** es una de las fases más críticas en cualquier proyecto de Machine Learning. Los modelos aprenden patrones a partir de los datos, por lo que la calidad de los datos de entrada determina directamente la calidad de las predicciones.

En este notebook trabajaremos con datos de **Lending Club** (préstamos P2P) para construir un modelo que prediga si un préstamo será pagado o entrará en impago (`loan_status`).

**Dataset:** ~80,000 préstamos con 93+ variables originales.

In [1]:
import pandas as pd

df_train = pd.read_csv("data/df_train_small.csv")

## Pipeline de Preprocesamiento

El preprocesamiento sigue un orden lógico donde cada paso depende del anterior:

1. **Selección inicial de variables** — Eliminar variables irrelevantes o que causan *data leakage*
2. **Tratamiento de valores nulos (NaN)** — Decidir qué hacer con los datos faltantes
3. **Procesamiento de variables categóricas** — Convertirlas a formato numérico
   - Baja cardinalidad → One-Hot Encoding
   - Alta cardinalidad (texto libre) → Técnicas especiales (Target Encoding, embeddings...)
4. **Procesamiento de variables numéricas** — Normalización, transformación, outliers
5. **Feature Engineering** — Crear nuevas variables a partir de las existentes
6. **Feature Selection** — Eliminar variables que no aportan información al modelo

> **Importante:** Todo el preprocesamiento se diseña SOLO sobre el conjunto de **entrenamiento**. Luego se aplican las mismas transformaciones al conjunto de test para evitar *data leakage*.

---
## 1. Selección Inicial de Variables

Antes de cualquier transformación, debemos seleccionar qué variables tienen sentido como **predictoras**. Se descartan:

- **Identificadores** (`id`, `member_id`): No tienen valor predictivo
- **Variables post-concesión** (`funded_amnt`, `funded_amnt_inv`): Generan **data leakage** porque solo se conocen después de aprobar el préstamo — exactamente lo que queremos predecir
- **Variables redundantes o irrelevantes** según criterio de negocio

Usamos un fichero Excel donde previamente se ha clasificado cada variable como posible predictora o no, junto con el motivo de exclusión.

In [2]:
raw_predictors_vars = pd.read_excel("data/variables_withoutExperts.xlsx")
raw_predictors_vars.head()

,variable,categoria,descripcion,posible_predictora,motivo_exclusion
0,id,identificador,Identificador unico del prestamo,no,Identificador sin valor predictivo
1,member_id,identificador,Identificador unico del miembro,no,Identificador sin valor predictivo
2,loan_amnt,solicitud,Importe del prestamo solicitado,si,NaN
3,funded_amnt,post_concesion,Importe total financiado,no,Data leakage: conocido post-aprobacion
4,funded_amnt_inv,post_concesion,Importe financiado por inversores,no,Data leakage: conocido post-aprobacion


In [3]:
raw_predictors_vars = ( raw_predictors_vars
                       .query("posible_predictora == 'si'")
                        .variable
                        .tolist())
raw_predictors_vars

['loan_amnt',
 'term',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'purpose',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'application_type',
 'annual_inc_joint',
 'dti_joint',
 'verification_status_joint',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'open_acc_6m',
 'open_act_il',
 'open_il_12m',
 'open_il_24m',
 'mths_since_rcnt_il',
 'total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m',
 'max_bal_bc',
 'all_util',
 'total_rev_hi_lim',
 'inq_fi',
 'total_cu_tl',
 'inq_last_12m',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',

Una vez identificadas las variables predictoras, construimos el dataset de trabajo seleccionando solo esas columnas más la **variable objetivo** (`loan_status`).

In [4]:
target_var = "loan_status"

df_train_filter_1 = df_train[ raw_predictors_vars + [target_var] ]
df_train_filter_1.head() 

,loan_amnt,term,emp_title,emp_length,home_ownership,annual_inc,verification_status,purpose,zip_code,addr_state,...,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,loan_status
0,8000.0,36 months,Supervisory Personal Property,10+ years,MORTGAGE,130000.0,Not Verified,home_improvement,225xx,VA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fully Paid
1,18000.0,36 months,Terminal Manager,7 years,MORTGAGE,106340.0,Source Verified,debt_consolidation,725xx,AR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fully Paid
2,6200.0,36 months,Journeyman Meatcutter,6 years,RENT,32000.0,Not Verified,credit_card,282xx,NC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fully Paid
3,10200.0,36 months,owner,10+ years,OWN,24000.0,Source Verified,debt_consolidation,351xx,AL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fully Paid
4,15000.0,36 months,Owner,10+ years,MORTGAGE,75000.0,Verified,debt_consolidation,465xx,IN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fully Paid


In [5]:
df_train_filter_1.shape

(80000, 94)

---
## 2. Tratamiento de Valores Nulos (Missing Values)

Los valores nulos son uno de los problemas más comunes en datos reales. La mayoría de algoritmos de ML **no pueden trabajar con NaN**, por lo que debemos tratarlos.

### Estrategia general

Primero analizamos el **porcentaje de nulos** de cada variable para decidir qué hacer:

| % de nulos | Estrategia |
|---|---|
| **> 99%** | Eliminar la variable (no aporta información útil) |
| **10% - 99%** | Imputar con valor especial: `-1` (numéricas) o `"DESCONOCIDO"` (categóricas) |
| **< 10%** | Imputar con **mediana** (numéricas) o **moda** (categóricas) |

> **¿Por qué mediana y no media?** La mediana es más robusta frente a outliers. Si hay valores extremos, la media se distorsiona pero la mediana se mantiene estable.

> **¿Por qué `-1` o `"DESCONOCIDO"` para >10%?** Cuando hay muchos nulos, imputar con mediana/moda introduciría un sesgo importante. Es mejor que el modelo aprenda que "dato desconocido" es una categoría en sí misma.

In [6]:
(df_train_filter_1.isnull().sum()/80000).sort_values(ascending=False).to_dict()

{'sec_app_mths_since_last_major_derog': 0.99705,
 'sec_app_revol_util': 0.99165,
 'sec_app_open_acc': 0.9915,
 'sec_app_mort_acc': 0.9915,
 'revol_bal_joint': 0.9915,
 'sec_app_earliest_cr_line': 0.9915,
 'sec_app_collections_12_mths_ex_med': 0.9915,
 'sec_app_chargeoff_within_12_mths': 0.9915,
 'sec_app_num_rev_accts': 0.9915,
 'sec_app_open_act_il': 0.9915,
 'sec_app_inq_last_6mths': 0.9915,
 'dti_joint': 0.9859625,
 'annual_inc_joint': 0.98595,
 'verification_status_joint': 0.98595,
 'mths_since_last_record': 0.82845,
 'mths_since_recent_bc_dlq': 0.760775,
 'mths_since_last_major_derog': 0.7346375,
 'il_util': 0.676475,
 'mths_since_recent_revol_delinq': 0.66205,
 'mths_since_rcnt_il': 0.63675,
 'open_rv_12m': 0.6272625,
 'open_acc_6m': 0.6272625,
 'open_il_24m': 0.6272625,
 'open_act_il': 0.6272625,
 'open_il_12m': 0.6272625,
 'inq_fi': 0.6272625,
 'total_bal_il': 0.6272625,
 'all_util': 0.6272625,
 'max_bal_bc': 0.6272625,
 'total_cu_tl': 0.6272625,
 'inq_last_12m': 0.6272625,
 'o

In [7]:
nulls_vars = (df_train_filter_1.isnull().sum()/80000).sort_values(ascending=False).to_frame(name="nulls_perc").reset_index()


### 2.1 Eliminación de variables con demasiados nulos

Las variables con más del **99%** de valores nulos no contienen información útil para el modelo. Son principalmente variables de solicitudes conjuntas (`sec_app_*`, `*_joint`) que solo aplican a un tipo muy minoritario de préstamos.

In [8]:
var_with_most_nulls = nulls_vars.query("nulls_perc >= 0.99150")["index"].tolist()
df_train_filter_2 = df_train_filter_1.drop(columns=var_with_most_nulls)
df_train_filter_2.shape

(80000, 83)

### 2.2 Imputación de valores nulos

Aplicamos la estrategia diferenciada según el porcentaje de nulos:

- **< 10% de nulos** → Imputamos con la **mediana** (numéricas) o la **moda** (categóricas), ya que el impacto en la distribución es mínimo
- **10% - 99% de nulos** → Imputamos con **-1** (numéricas) o **"DESCONOCIDO"** (categóricas), creando una categoría explícita para datos faltantes

In [9]:
nulls_10_perc = nulls_vars.query("nulls_perc < 0.10")["index"].tolist()
nulls_more_10_perc = nulls_vars.query("nulls_perc >= 0.10 and nulls_perc < 0.99150")["index"].tolist()

categoric_vars = df_train_filter_2.select_dtypes(include="object").columns.tolist()
numeric_vars = df_train_filter_2.select_dtypes(include="number").columns.tolist()

/var/folders/_0/0zwytxv51nb3c6wxtthk3bwh0000gn/T/ipykernel_19563/3048532664.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoric_vars = df_train_filter_2.select_dtypes(include="object").columns.tolist()


In [10]:
for var in nulls_10_perc:
    if var in categoric_vars:
        df_train_filter_2[var] = df_train_filter_2[var].fillna(df_train_filter_2[var].mode()[0])
    else:
        df_train_filter_2[var] = df_train_filter_2[var].fillna(df_train_filter_2[var].median())

for var in nulls_more_10_perc:
    if var in categoric_vars:
        df_train_filter_2[var] = df_train_filter_2[var].fillna("DESCONOCIDO")
    else:
        df_train_filter_2[var] = df_train_filter_2[var].fillna(-1)

Verificamos que no quedan valores nulos en el dataset tras la imputación:

In [11]:
df_train_filter_2.isnull().sum().sum()

np.int64(0)

---
## 3. Procesamiento de Variables Categóricas

Los modelos de ML trabajan con números, no con texto. Necesitamos convertir las variables categóricas a formato numérico. La estrategia depende de la **cardinalidad** (número de valores únicos):

| Cardinalidad | Ejemplo | Estrategia |
|---|---|---|
| **Baja** (< ~15 valores) | `purpose`, `home_ownership` | **One-Hot Encoding** |
| **Media** (~15-100 valores) | `addr_state` | One-Hot o **Target Encoding** |
| **Alta** (> 100 valores) | `emp_title`, `zip_code` | **Target Encoding**, embeddings o eliminar |

Primero, analicemos la cardinalidad de nuestras variables categóricas:

In [12]:
categoric_vars = ( df_train_filter_2
                  .select_dtypes(include="object")
                  .columns
                  .tolist() )

categoric_vars_cardinality = ( df_train_filter_2[categoric_vars]
                              .nunique()
                              .sort_values(ascending=False)
                              .to_frame(name="cardinality")
                              .reset_index())
categoric_vars_cardinality

/var/folders/_0/0zwytxv51nb3c6wxtthk3bwh0000gn/T/ipykernel_19563/1468478250.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include="object")


,index,cardinality
0,emp_title,36885
1,zip_code,872
2,earliest_cr_line,634
3,addr_state,50
4,purpose,14
5,emp_length,11
6,home_ownership,6
7,verification_status_joint,4
8,verification_status,3
9,term,2


### 3.1 Decisiones sobre variables categóricas de alta cardinalidad

Observamos que:

- **`emp_title`** tiene **36,885 valores únicos** → Demasiada cardinalidad para One-Hot. Se tratará con técnicas especiales (Target Encoding, NLP) o se eliminará
- **`zip_code`** tiene **872 valores** → Redundante con `addr_state` (50 estados). Eliminamos `zip_code` para reducir dimensionalidad sin perder información geográfica relevante
- **`earliest_cr_line`** tiene **634 valores** → Son fechas almacenadas como texto. Requieren conversión a formato fecha

In [13]:
df_train_filter_3 = df_train_filter_2.drop(columns=['zip_code'])

In [14]:
df_train_filter_2['earliest_cr_line'].head(20)

0     Nov-1990
1     Jun-1975
2     Sep-2004
3     Oct-2000
4     Nov-1989
5     Jun-1998
6     Sep-2006
7     Jun-1987
8     Sep-1990
9     May-2001
10    Oct-2003
11    Dec-1999
12    Oct-2000
13    May-2004
14    Aug-2007
15    Jul-2008
16    Nov-2005
17    Sep-2004
18    Aug-2000
19    Sep-1998
Name: earliest_cr_line, dtype: str

### 3.2 Variables de fecha almacenadas como texto

`earliest_cr_line` contiene fechas en formato `"Mes-Año"` (ej: `Nov-1990`). Para que el modelo pueda utilizarla, necesitamos:

1. Convertirla a tipo `datetime`
2. Extraer features útiles como la **antigüedad crediticia en meses**

Esto es un ejemplo de **Feature Engineering**: transformar datos crudos en variables que el modelo pueda interpretar mejor.

> **Próximos pasos:** Continuar con el encoding de categóricas de baja cardinalidad (One-Hot Encoding), normalización de numéricas y feature engineering.